In [ ]:
import autogen
import chromadb
from chromadb.config import Settings
from datetime import datetime

# Initialize ChromaDB client
chroma_client = chromadb.Client(Settings(allow_reset=True))
collection = chroma_client.create_collection("conversation_memory")

# Configure agents
config_list = [
    {
        "model": "llama3.2",
        "base_url": "http://localhost:11434/v1",
        'api_key': 'ollama',
    },
]
# Assistant agent configuration
assistant = autogen.AssistantAgent(
    name="assistant",
    llm_config={
        "config_list": config_list,
        "cache_seed": 42
    }
)

# User proxy configuration
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=10
)

def save_to_memory(messages):
    timestamp = datetime.now().isoformat()
    collection.add(
        documents=[str(msg["content"]) for msg in messages],
        metadatas=[{"timestamp": timestamp, "role": msg["role"]} for msg in messages],
        ids=[f"{timestamp}_{i}" for i in range(len(messages))]
    )

def get_context_from_memory(query, n_results=5):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results['documents'][0]

# Initiate a conversation with memory
def chat_with_memory(user_message):
    # Get relevant context from memory
    context = get_context_from_memory(user_message)
    
    # Add context to the message
    message_with_context = f"Context from previous conversations: {context}\n\nCurrent query: {user_message}"
    
    # Start the conversation
    user_proxy.initiate_chat(
        assistant,
        message=message_with_context
    )
    
    # Save the conversation to memory
    save_to_memory(user_proxy.chat_messages[assistant])



In [8]:
import autogen
import chromadb
import uuid
from datetime import datetime
from chromadb.config import Settings

# Generate unique conversation ID
CONVERSATION_ID = str(uuid.uuid4())

# Initialize ChromaDB
chroma_client = chromadb.Client(Settings(allow_reset=True))
collection = chroma_client.get_or_create_collection("conversation_history")

# Agent configurations
config_list = [
    {
        "model": "llama3.2",
        "base_url": "http://localhost:11434/v1",
        'api_key': 'ollama',
    },
]

llm_config = {
    "config_list": config_list,
    "seed": 42,
    "request_timeout": 120
}

# Create agents
assistant = autogen.AssistantAgent(
    name="assistant",
    llm_config=llm_config,
    system_message="You are a helpful AI assistant. Use previous conversation context when relevant."
)

user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=10,
    code_execution_config=False
)

def save_conversation(messages, conversation_id=CONVERSATION_ID):
    timestamp = datetime.now().isoformat()
    
    documents = [str(msg["content"]) for msg in messages]
    metadatas = [{
        "timestamp": timestamp,
        "role": msg["role"],
        "conversation_id": conversation_id
    } for msg in messages]
    ids = [f"{conversation_id}_{timestamp}_{i}" for i in range(len(messages))]
    
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

def get_conversation_history(query, n_results=5):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results['documents'][0]

def initiate_chat(message):
    print(f"Conversation ID: {CONVERSATION_ID}")
    
    # Get relevant history
    history = get_conversation_history(message)
    context_message = f"Previous relevant context:\n{history}\n\nCurrent query: {message}"
    
    # Start chat
    user_proxy.initiate_chat(
        assistant,
        message=context_message
    )
    
    # Save conversation
    save_conversation(user_proxy.chat_messages[assistant])

if __name__ == "__main__":
    user_input = "What are the best practices for Python error handling?"
    initiate_chat(user_input)

Conversation ID: 1c5b29d6-4934-48d2-8e4a-79e0aa52e364


C:\Users\anoop\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz:   6%|▌         | 4.84M/79.3M [00:46<11:53, 110kiB/s]    


KeyboardInterrupt: 

In [ ]:
import autogen
import chromadb
import uuid
import openai
from datetime import datetime
from chromadb.config import Settings
from typing import List, Dict

# OpenAI and ChromaDB Setup
OPENAI_API_KEY = "your-openai-api-key"
openai.api_key = OPENAI_API_KEY

# Initialize ChromaDB with OpenAI embeddings
chroma_client = chromadb.Client(Settings(
    chroma_db_impl="duckdb+parquet",
    persist_directory="./memory_store"
))
collection = chroma_client.get_or_create_collection(
    name="conversation_memory",
    embedding_function=lambda texts: [
        openai.Embedding.create(input=text, model="text-embedding-3-small")["data"][0]["embedding"]
        for text in texts
    ]
)

# Agent configurations
config_list = [
    {
        "model": "gpt-3.5-turbo",
        "api_key": OPENAI_API_KEY
    }
]

llm_config = {
    "config_list": config_list,
    "seed": 42
}

# Create agents
assistant = autogen.AssistantAgent(
    name="assistant",
    llm_config=llm_config,
    system_message="You are a helpful AI assistant. Use previous conversation context when relevant."
)

user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=10
)

def save_to_memory(messages: List[Dict], conversation_id: str):
    timestamp = datetime.now().isoformat()
    
    documents = [str(msg["content"]) for msg in messages]
    metadatas = [{
        "timestamp": timestamp,
        "role": msg["role"],
        "conversation_id": conversation_id
    } for msg in messages]
    ids = [f"{conversation_id}_{timestamp}_{i}" for i in range(len(messages))]
    
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

def get_relevant_context(query: str, n_results: int = 5) -> List[str]:
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results['documents'][0]

def chat_with_memory(user_message: str):
    conversation_id = str(uuid.uuid4())
    print(f"Starting conversation: {conversation_id}")
    
    # Get context from similar conversations
    context = get_relevant_context(user_message)
    enhanced_message = f"Context from previous conversations:\n{context}\n\nCurrent query: {user_message}"
    
    # Initiate chat with context
    user_proxy.initiate_chat(
        assistant,
        message=enhanced_message
    )
    
    # Save conversation to memory
    save_to_memory(user_proxy.chat_messages[assistant], conversation_id)
    return conversation_id

if __name__ == "__main__":
    question = "What are the best practices for Python error handling?"
    conversation_id = chat_with_memory(question)

In [5]:
import autogen
import chromadb
import uuid
from datetime import datetime
from chromadb.config import Settings
import requests
import numpy as np
from typing import List, Dict

# Nomic embeddings function using Ollama
def get_embedding(text: str) -> List[float]:
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": "nomic-embed-text", "prompt": text}
    )
    return response.json()["embedding"]

# Initialize ChromaDB with Nomic embeddings
chroma_client = chromadb.Client(Settings(allow_reset=True))

class NomicEmbeddingFunction:
    def __call__(self, input: List[str]) -> List[List[float]]:
        return [get_embedding(text) for text in input]

embedding_function = NomicEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="conversation_memory",
    embedding_function=embedding_function
)

# Agent configurations
config_list = [
    {
        "model": "llama3.2",
        "base_url": "http://localhost:11434/v1",
        'api_key': 'ollama',
    },
]

llm_config = {
    "config_list": config_list,
    "seed": 42
}

# Create agents
assistant = autogen.AssistantAgent(
    name="assistant",
    llm_config=llm_config,
    system_message="You are a helpful AI assistant. Use previous conversation context when relevant."
)

user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=1,
    code_execution_config=False
)

def save_to_memory(messages: List[Dict], conversation_id: str):
    timestamp = datetime.now().isoformat()
    
    documents = [str(msg["content"]) for msg in messages]
    metadatas = [{
        "timestamp": timestamp,
        "role": msg["role"],
        "conversation_id": conversation_id
    } for msg in messages]
    ids = [f"{conversation_id}_{timestamp}_{i}" for i in range(len(messages))]
    
    try:
        collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
    except Exception as e:
        print(f"Error saving to memory: {e}")

def get_relevant_context(query: str, n_results: int = 5) -> List[str]:
    try:
        results = collection.query(
            query_texts=[query],
            n_results=n_results
        )
        return results['documents'][0]
    except Exception as e:
        print(f"Error retrieving context: {e}")
        return []

def chat_with_memory(user_message: str):
    conversation_id = str(uuid.uuid4())
    print(f"Starting conversation: {conversation_id}")
    
    context = get_relevant_context(user_message)
    enhanced_message = f"Context from previous conversations:\n{context}\n\nCurrent query: {user_message}"
    
    user_proxy.initiate_chat(
        assistant,
        message=enhanced_message
    )
    
    save_to_memory(user_proxy.chat_messages[assistant], conversation_id)
    return conversation_id

if __name__ == "__main__":
    # Make sure Ollama is running with nomic-embed-text model
    question = "What are the best practices for Python error handling?"
    conversation_id = chat_with_memory(question)

Starting conversation: 2d2834ae-1a1e-4a9f-b7f3-5ad782765c24
user_proxy (to assistant):

Context from previous conversations:
[]

Current query: What are the best practices for Python error handling?

--------------------------------------------------------------------------------
assistant (to user_proxy):

No specific context to draw from, but I can provide you with some widely accepted best practices for Python error handling:

1. **Usetry-except blocks**: Wrap potentially error-prone code in try blocks, and use except blocks to catch and handle specific exceptions.
2. **Be specific when catching exceptions**: Catch broad exceptions (e.g., `Exception`) instead of narrow ones (e.g., `RuntimeError`). This allows you to handle more common issues before resorting to bare `except` blocks.
3. **Log errors**: Store error messages for later analysis or debugging using logging frameworks like Python's built-in `logging` module or third-party libraries like Loguru.
4. **Provide informative err

KeyboardInterrupt: 